<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Projekt-Templates & MVP
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Dieses Notebook macht den Transfer auf **eigene** Projekte: wiederverwendbare Architektur-Templates (Research, Analyse, Support) und die Frage, wann ein Multi-Agent-System als **MVP** gilt.



**Praxisanker:** Mara Vogt kann wiederkehrende Briefings mit einer Vorlage einheitlich aufbauen, statt jedes Mal Struktur, Quellenfelder und Prüfschritte neu zu formulieren.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "m27-projekt-templates"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "tags": ["m27", "pipeline"],
    "metadata": {"notebook": "m27", "version": "1.0"}
}


# 1 | Übersicht
---

Dieses Notebook abstrahiert von einer konkreten Multi-Agent-Pipeline: drei **Architektur-Templates** (A: Research/Recherche, B: Datenanalyse, C: Support/FAQ) als Ausgangspunkt für eigene Multi-Agent-Systeme, gefolgt von einer **MVP-Checkliste**, die klärt, wann ein System lauffähig, getestet und dokumentiert genug für einen ersten produktiven Einsatz ist.

# 2 | Transfer: Projekt-Templates
---


Dieser Abschnitt ist ein Transfer-Baukasten. Die eigentliche Pipeline bleibt Security → Research → Writing → Judge; die Templates zeigen nur, wie dasselbe Muster auf andere Projekttypen übertragen wird.

Drei bewährte Templates decken die häufigsten Multi-Agent-Anwendungsfälle ab.
Jedes Template ist ein vollständiger Supervisor-Graph – direkt einsetzbar.

<p><font color='black' size="5">Template A – Content Pipeline</font></p>

**Einsatz:** Artikel, Berichte, Zusammenfassungen, Blogposts  
**Agents:** Recherche → Schreiben  
**Kernidee:** Fakten zuerst sammeln, dann strukturiert verfassen.


In [168]:
#@markdown   <p><font size="4" color='green'>  Template A – Content Pipeline</font> </br></p>

diag_a = '''
%%{init: {'theme':'light'}}%%
flowchart TD
    START([START]) --> SUP["🎯 Supervisor\nContent-Koordinator"]

    SUP -->|"research"| R["🔍 Recherche-Agent\nFakten sammeln"]
    SUP -->|"schreiben"| W["✍️ Schreib-Agent\nText erstellen"]
    SUP -->|"FINISH"| END([END])

    R -->|"AIMessage\n(name=Recherche)"| SUP
    W -->|"AIMessage\n(name=Schreiben)"| SUP

    R --- RT1["📚 search_reference_sources"]
    W --- WT1["📋 create_briefing_outline"]
    W --- WT2["🔢 count_briefing_words"]

    style SUP fill:#FF9800,color:#000
    style R   fill:#2196F3,color:#fff
    style W   fill:#4CAF50,color:#fff
    style RT1 fill:#1565C0,color:#fff
    style WT1 fill:#2E7D32,color:#fff
    style WT2 fill:#2E7D32,color:#fff
'''
mermaid(diag_a, width=700)

In [ ]:
#@markdown   <p><font size="4" color='green'>  Template A – Implementierung</font> </br></p>

# ── Template A: Content Pipeline ──────────────────────────────────────────
@tool
def search_reference_sources(topic: str) -> str:
    '''Sucht Informationen zu einem Thema (simuliert Wikipedia).'''
    wissen = {
        "langchain": "LangChain: Framework für LLM-Anwendungen. Chains, Agents, Tools, LCEL.",
        "langgraph": "LangGraph: Zustandsbasierte Agenten-Graphen mit StateGraph und Checkpointing.",
        "rag":       "RAG: Retrieval-Augmented Generation kombiniert Vektorsuche mit LLM-Generierung.",
        "langsmith": "LangSmith: Tracing, Evaluation und Monitoring für LLM-Pipelines.",
    }
    key = topic.lower().split()[0].rstrip("?.,:")
    return wissen.get(key, f'Allgemeine Informationen zu "{topic}" gefunden.')

@tool
def create_briefing_outline(topic: str, stichpunkte: str) -> str:
    '''Erstellt eine strukturierte Gliederung für einen Artikel.'''
    return (
        f'Gliederung zu "{topic}":\n'
        f'1. Einleitung\n2. Kernkonzept\n3. Anwendung\n4. Fazit\n\n'
        f'Basis: {stichpunkte[:120]}'
    )

@tool
def count_briefing_words(text: str) -> str:
    '''Zählt Wörter in einem Text.'''
    n = len(text.split())
    return f'{n} Wörter | {len(text)} Zeichen'

class ContentState(TypedDict):
    topic: str
    research: str
    outline: str
    article: str

def recherche_a_node(state: ContentState) -> dict:
    return {"research": search_reference_sources.invoke({"topic": state["topic"]})}

def schreiben_a_node(state: ContentState) -> dict:
    outline = create_briefing_outline.invoke({"topic": state["topic"], "stichpunkte": state["research"]})
    article = (
        f"Kurzartikel zu {state['topic']}\n"
        f"Recherche: {state['research']}\n"
        f"{outline}\n"
        "Fazit: Das Template trennt Faktenrecherche und Texterstellung klar voneinander."
    )
    return {"outline": outline, "article": article + "\n" + count_briefing_words.invoke({"text": article})}

content_a_builder = StateGraph(ContentState)
content_a_builder.add_node("research", recherche_a_node)
content_a_builder.add_node("schreiben", schreiben_a_node)
content_a_builder.add_edge(START, "research")
content_a_builder.add_edge("research", "schreiben")
content_a_builder.add_edge("schreiben", END)
content_agent = content_a_builder.compile()

print("✅ Template A: Content Pipeline bereit (StateGraph: Recherche → Schreiben)")


In [ ]:
#@markdown   <p><font size="4" color='green'>  Template A – Demo</font> </br></p>

result_a = content_agent.invoke(
    {"topic": "LangChain", "research": "", "outline": "", "article": ""},
    config={
        "recursion_limit": 10,
        "run_name": "m27_kap2_templatea_demo",
        "tags": ["m27", "template-a", "content-pipeline"],
    },
)
mprint("### 📝 Template A – Content Pipeline\n")
mprint(result_a["article"])



<p><font color='black' size="5">Template B – Analyse Pipeline</font></p>

**Einsatz:** Datenanalyse, Code-Reviews, Auswertungen  
**Agents:** Erfassen → Analysieren  
**Kernidee:** Rohdaten aufnehmen, dann auswerten und interpretieren.


In [171]:
#@markdown   <p><font size="4" color='green'>  Template B – Analyse Pipeline</font> </br></p>

diag_b = '''
%%{init: {'theme':'light'}}%%
flowchart TD
    START([START]) --> SUP["🎯 Supervisor\nAnalyse-Koordinator"]

    SUP -->|"erfassen"| E["📥 Erfassungs-Agent\nDaten aufnehmen"]
    SUP -->|"analysieren"| A["🔬 Analyse-Agent\nAuswerten"]
    SUP -->|"FINISH"| END([END])

    E -->|"AIMessage\n(name=Erfassen)"| SUP
    A -->|"AIMessage\n(name=Analyse)"| SUP

    E --- ET1["📊 daten_laden"]
    A --- AT1["▶️ python_ausfuehren"]
    A --- AT2["🔍 muster_suchen"]

    style SUP fill:#FF9800,color:#000
    style E   fill:#9C27B0,color:#fff
    style A   fill:#F44336,color:#fff
    style ET1 fill:#6A1B9A,color:#fff
    style AT1 fill:#B71C1C,color:#fff
    style AT2 fill:#B71C1C,color:#fff
'''
mermaid(diag_b, width=700)

In [ ]:
#@markdown   <p><font size="4" color='green'>  Template B – Implementierung</font> </br></p>

import builtins
import re

# ── Template B: Analyse Pipeline ──────────────────────────────────────────
@tool
def daten_laden(quelle: str) -> str:
    '''Lädt Rohdaten aus einer Quelle (simuliert).'''
    daten = {
        "sales":   "Q1: 120k€, Q2: 145k€, Q3: 98k€, Q4: 167k€",
        "traffic": "Jan: 12.400 Besucher, Feb: 9.800, Mar: 15.600",
        "fehler":  "TypeError: 42x, KeyError: 18x, TimeoutError: 7x",
    }
    key = quelle.lower().split()[0].rstrip("?.,:")
    return daten.get(key, f'Daten aus "{quelle}": [100, 200, 150, 300, 250]')

@tool
def python_ausfuehren(ausdruck: str) -> str:
    '''Wertet einen einfachen Python-Ausdruck sicher aus.'''
    try:
        erlaubt = {k: getattr(builtins, k, None)
                   for k in ('sum', 'max', 'min', 'len', 'round', 'abs', 'sorted')}
        erlaubt = {k: v for k, v in erlaubt.items() if v is not None}
        return f'Ergebnis: {eval(ausdruck, {"__builtins__": erlaubt})}'
    except Exception as e:
        return f'Auswertungsfehler: {e}'

@tool
def muster_suchen(text: str, stichwort: str = "max") -> str:
    '''Sucht nach einem Stichwort im Text und gibt passende Zeilen zurück.'''
    treffer = [z for z in text.split(",") if stichwort.lower() in z.lower()]
    return f'Treffer für "{stichwort}": {treffer[0].strip() if treffer else "nicht gefunden"}'

template_b_tool_names = [daten_laden.name, python_ausfuehren.name, muster_suchen.name]
assert all(name.replace('_', '').replace('-', '').isalnum() for name in template_b_tool_names), template_b_tool_names

class AnalysisState(TypedDict):
    query: str
    raw_data: str
    analysis: str
    report: str

def _template_b_quelle(query: str) -> str:
    text = query.lower()
    if "traffic" in text or "besucher" in text:
        return "traffic"
    if "fehler" in text or "error" in text:
        return "fehler"
    return "sales"

def erfassen_b_node(state: AnalysisState) -> dict:
    quelle = _template_b_quelle(state["query"])
    raw_data = daten_laden.invoke({"quelle": quelle})
    return {"raw_data": raw_data}

def analysieren_b_node(state: AnalysisState) -> dict:
    werte = [(q, int(v)) for q, v in re.findall(r"(Q[1-4]):\s*(\d+)k", state["raw_data"])]
    if werte:
        bestes_quartal, bester_wert = max(werte, key=lambda item: item[1])
        berechnung = python_ausfuehren.invoke({"ausdruck": f"max({[v for _, v in werte]})"})
        muster = muster_suchen.invoke({"text": ", ".join(f"{q}: {v}k" for q, v in werte), "stichwort": bestes_quartal})
        analysis = f"Bestes Quartal: {bestes_quartal} mit {bester_wert}k€. {berechnung}. {muster}."
    else:
        analysis = "Keine Quartalswerte erkannt; Daten wurden erfasst, aber nicht quantitativ ausgewertet."
    return {"analysis": analysis}

def bericht_b_node(state: AnalysisState) -> dict:
    report = (
        "Analyse-Bericht\n"
        f"Anfrage: {state['query']}\n"
        f"Rohdaten: {state['raw_data']}\n"
        f"Ergebnis: {state['analysis']}"
    )
    return {"report": report}

analyse_b_builder = StateGraph(AnalysisState)
analyse_b_builder.add_node("erfassen", erfassen_b_node)
analyse_b_builder.add_node("analysieren", analysieren_b_node)
analyse_b_builder.add_node("report", bericht_b_node)
analyse_b_builder.add_edge(START, "erfassen")
analyse_b_builder.add_edge("erfassen", "analysieren")
analyse_b_builder.add_edge("analysieren", "report")
analyse_b_builder.add_edge("report", END)
analyse_agent = analyse_b_builder.compile()

print("✅ Template B: Analyse Pipeline bereit (StateGraph: Erfassen → Analysieren → Bericht)")


In [ ]:
#@markdown   <p><font size="4" color='green'>  Template B – Demo</font> </br></p>

result_b = analyse_agent.invoke(
    {
        "query": "Analysiere die Sales-Daten und finde das beste Quartal.",
        "raw_data": "",
        "analysis": "",
        "report": "",
    },
    config={
        "recursion_limit": 10,
        "run_name": "m27_kap2_templateb_demo",
        "tags": ["m27", "template-b", "analysis-pipeline"],
    },
)
mprint("### 📊 Template B – Analyse Pipeline\n")
mprint(result_b["report"])



<p><font color='black' size="5">Template C – Support Pipeline</font></p>

**Einsatz:** Ticket-Bearbeitung, FAQ-Systeme, Kundenservice  
**Agents:** Klassifizieren → Lösen  
**Kernidee:** Anfrage kategorisieren, dann zielgerichtet beantworten.

In [174]:
#@markdown   <p><font size="4" color='green'>  Template C – Support Pipeline</font> </br></p>

diag_c = '''
%%{init: {'theme':'dark'}}%%
flowchart TD
    START([START]) --> SUP["🎯 Supervisor\nSupport-Koordinator"]

    SUP -->|"klassifizieren"| K["🏷️ Klassifizier-Agent\nKategorie bestimmen"]
    SUP -->|"loesen"| L["🔧 Lösungs-Agent\nAntwort generieren"]
    SUP -->|"FINISH"| END([END])

    K -->|"AIMessage\n(name=Klassifizierung)"| SUP
    L -->|"AIMessage\n(name=Loesung)"| SUP

    K --- KT1["🗂️ ticket_kategorisieren"]
    L --- LT1["📖 faq_suchen"]
    L --- LT2["✅ antwort_validieren"]

    style SUP fill:#FF9800,color:#000
    style K   fill:#00BCD4,color:#000
    style L   fill:#8BC34A,color:#000
    style KT1 fill:#006064,color:#fff
    style LT1 fill:#33691E,color:#fff
    style LT2 fill:#33691E,color:#fff
'''
mermaid(diag_c, width=700)

In [ ]:
#@markdown   <p><font size="4" color='green'>  Template C – Implementierung</font> </br></p>

# ── Template C: Support Pipeline ──────────────────────────────────────────
@tool
def ticket_kategorisieren(ticket: str) -> str:
    '''Kategorisiert ein Support-Ticket nach Typ und Priorität.'''
    text = ticket.lower()
    if any(w in text for w in ["error", "fehler", "absturz", "funktioniert nicht", "bug"]):
        return "Kategorie: Technisch | Priorität: Hoch"
    if any(w in text for w in ["rechnung", "zahlung", "preis", "kosten", "abonnement"]):
        return "Kategorie: Billing | Priorität: Mittel"
    return "Kategorie: Allgemein | Priorität: Niedrig"

@tool
def faq_suchen(frage: str) -> str:
    '''Sucht eine passende FAQ-Antwort.'''
    faq = {
        "passwort":    "Passwort zurücksetzen: Einstellungen → Sicherheit → Passwort ändern.",
        "rechnung":    "Rechnungen: Konto → Abrechnungen → Rechnungshistorie.",
        "installieren": "Installation: Setup-Datei laden, als Administrator ausführen.",
        "fehler":      "Fehler: App neu starten, Cache leeren, ggf. Support kontaktieren.",
    }
    for key, answer in faq.items():
        if key in frage.lower():
            return answer
    return f'Kein FAQ-Treffer für "{frage[:50]}". Weiterleitung an Support-Team.'

@tool
def antwort_validieren(answer: str) -> str:
    '''Prüft, ob eine Support-Antwort vollständig und hilfreich ist.'''
    if len(answer) < 20:
        return f'⚠️ Antwort zu kurz — bitte ergänzen.'
    return f'✅ Validiert ({len(answer)} Zeichen): {answer[:80]}...'

class SupportState(TypedDict):
    query: str
    category: str
    faq_answer: str
    answer: str

def klassifizieren_c_node(state: SupportState) -> dict:
    return {"category": ticket_kategorisieren.invoke({"ticket": state["query"]})}

def loesen_c_node(state: SupportState) -> dict:
    faq_answer = faq_suchen.invoke({"frage": state["query"]})
    validierung = antwort_validieren.invoke({"answer": faq_answer})
    answer = f"{state['category']}\nEmpfehlung: {faq_answer}\n{validierung}"
    return {"faq_answer": faq_answer, "answer": answer}

support_c_builder = StateGraph(SupportState)
support_c_builder.add_node("klassifizieren", klassifizieren_c_node)
support_c_builder.add_node("loesen", loesen_c_node)
support_c_builder.add_edge(START, "klassifizieren")
support_c_builder.add_edge("klassifizieren", "loesen")
support_c_builder.add_edge("loesen", END)
support_agent = support_c_builder.compile()

print("✅ Template C: Support Pipeline bereit (StateGraph: Klassifizieren → Lösen)")


In [ ]:
#@markdown   <p><font size="4" color='green'>  Template C – Demo</font> </br></p>

result_c = support_agent.invoke(
    {
        "query": "Ich bekomme einen Fehler beim Starten der App. Was kann ich tun?",
        "category": "",
        "faq_answer": "",
        "answer": "",
    },
    config={
        "recursion_limit": 10,
        "run_name": "m27_kap2_templatec_demo",
        "tags": ["m27", "template-c", "support-pipeline"],
    },
)
mprint("### 🎫 Template C – Support Pipeline\n")
mprint(result_c["answer"])


**Template-Vergleich:**

| | Template A | Template B | Template C |
|---|---|---|---|
| **Name** | Content Pipeline | Analyse Pipeline | Support Pipeline |
| **Agents** | Recherche, Schreiben | Erfassen, Analysieren | Klassifizieren, Lösen |
| **Input** | Thema / Frage | Daten / Code | Anfrage / Ticket |
| **Output** | Strukturierter Text | Auswertung / Bericht | Kategorisierte Antwort |
| **Typisches Tool** | search_reference_sources | python_ausfuehren | faq_suchen |
| **Komplexität** | ⭐⭐ | ⭐⭐⭐ | ⭐⭐ |

<p><font color='black' size="5">
Template wählen
</font></p>

Die Wahl des richtigen Templates hängt vom **Output-Typ** ab:

- **Text/Inhalt erzeugen** → Template A (Content)
- **Daten/Code auswerten** → Template B (Analyse)
- **Anfragen bearbeiten** → Template C (Support)

Nutze den Entscheidungsbaum als erste Orientierung:

In [177]:
#@markdown   <p><font size="4" color='green'>  Template-Entscheidungsbaum</font> </br></p>

diag_e = '''
%%{init: {'theme':'light'}}%%
flowchart TD
    START(["❓ Mein Anwendungsfall"]) --> Q1{"Geht es um\nText-Erstellung?"}

    Q1 -->|"JA"| Q2{"Brauche ich\nFakten-Recherche?"}
    Q1 -->|"NEIN"| Q3{"Geht es um\nDatenauswertung?"}

    Q2 -->|"JA"| A["✅ Template A\nContent Pipeline"]
    Q2 -->|"NEIN"| A2["✅ Template A\n(nur Schreib-Agent)"]

    Q3 -->|"JA"| B["✅ Template B\nAnalyse Pipeline"]
    Q3 -->|"NEIN"| Q4{"Bearbeite ich\nAnfragen?"}

    Q4 -->|"JA"| C["✅ Template C\nSupport Pipeline"]
    Q4 -->|"NEIN"| CUSTOM["⚙️ Eigenes Template\nSiehe Aufgaben 5"]

    style A      fill:#4CAF50,color:#fff
    style A2     fill:#4CAF50,color:#fff
    style B      fill:#F44336,color:#fff
    style C      fill:#00BCD4,color:#000
    style CUSTOM fill:#FF9800,color:#000
    style Q1     fill:#37474F,color:#fff
    style Q2     fill:#37474F,color:#fff
    style Q3     fill:#37474F,color:#fff
    style Q4     fill:#37474F,color:#fff
'''
mermaid(diag_e, width=780)

# 3 | MVP-Definition
---

Ein **MVP (Minimum Viable Product)** für ein Multi-Agent-System ist lauffähig,
getestet und klar dokumentiert – aber noch nicht production-ready.



<p><font color='black' size="5">MVP-Checkliste</font></p>

| Kriterium | Beschreibung | Status Integration_Pipeline |
|-----------|-------------|---------------|
| ✅ **Läuft durch** | Graph endet ohne Fehler | Getestet |
| ✅ **Iterations-Schutz** | `max_iter` + `recursion_limit` | Implementiert |
| ✅ **Strukturiertes Routing** | `with_structured_output` | Implementiert |
| ✅ **Prompts extern** | `load_prompt()` statt Inline | Implementiert |
| ✅ **LangSmith-Tracing** | `run_name` + `tags` | Konfiguriert |
| ⚠️ **Fehlerbehandlung** | Try/Except in Worker-Nodes | Fehlt noch |
| ⚠️ **Checkpointing** | SQLite-Persistenz | Fehlt noch |
| ⚠️ **Tests** | Unit-Tests für Tools | Fehlt noch |
| ❌ **UI** | Gradio / Streamlit | Nicht in MVP |



<p><font color='black' size="5">Was kommt nach dem MVP?</font></p>

- ***OpenAI Agent Builder:*** Alternativer Ansatz ohne LangGraph-Boilerplate
- ***Agentic RAG:*** RAG-Agenten mit adaptivem Retrieval und Multi-Hop-Reasoning
- ***Gradio UI für Agenten:*** Web-Interface direkt für den eigenen Agenten
- ***Production Deployment:*** Fehlerbehandlung, Tests, Docker, Monitoring

<p><font color='darkblue' size="4">💡 <b>Tipp</b></font></p>

Starte immer mit dem MVP. Es ist besser, ein funktionierendes System mit 2 Agents zu haben als ein perfektes System das nie fertig wird.

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen — eigene Herausforderungen sind ausdrücklich willkommen.

**Hinweis zur Lösungshilfe:**
> Generative KI darf und soll im Kurs auch als Lernunterstützung genutzt werden — z. B. Gemini in Google Colab, um Fehlermeldungen zu verstehen, Teilschritte zu klären oder Code-Varianten zu prüfen.

**Grundlagen** Baue für einen eigenen Use Case (Research Report, Evaluation
Dashboard oder Review Gate) ein Multi-Agent-System aus Template A, B oder C.

Pflichtbestandteile sind:

- ein Supervisor,
- mindestens ein spezialisierter Worker,
- eine nachvollziehbare, deterministische Qualitätsprüfung für das Ergebnis.

Nutze dafür vorhandene Bausteine wie antwort_validieren aus Template C oder
formuliere für Template A/B eine einfache Prüfung auf Nicht-Leerheit,
Mindestlänge und Quellen-/Begründungshinweis. Ein LLM-as-Judge ist optional
und wird erst im Integrations-Notebook M26 vertieft.


In [ ]:
# Grundlagen: Eigenes System aus Template A, B oder C planen
# 1. USE_CASE waehlen: research_report, evaluation_dashboard oder review_gate
# 2. Supervisor und Worker aus einem Template als Ausgangspunkt bestimmen
# 3. Eine deterministische Qualitaetspruefung festlegen:
#    - Template C: vorhandenes Tool antwort_validieren nutzen
#    - Template A/B: Nicht-Leerheit, Mindestlaenge und Quellen-/Begruendungshinweis pruefen
# 4. mein_system_plan als Dictionary mit Rollen, Nodes und Pruefkriterium ausgeben
USE_CASE = "research_report"
mein_system_plan = {
    "use_case": USE_CASE,
    "template": "A",
    "rollen": ["supervisor", "worker", "quality_check"],
    "pruefkriterium": "Ergebnis nicht leer und mit Quellen- oder Begruendungshinweis",
}
print(mein_system_plan)


**Vertiefung** Ersetze die Worker-Logik deines gewählten Templates durch eine
fehlertolerante Node-Fabrik mit Retry-Logik und sauberem Fallback bei Timeout
oder API-Fehler. Verwende die vorhandene State-Struktur des Templates und
speichere mindestens status (ok oder fallback) sowie eine verständliche
Fehlerliste im State.


In [ ]:
# Vertiefung: Fehlertolerante Worker-Node
# Startpunkt: bestehenden Worker-Node (z.B. aus Template A/B/C) als Basis nutzen

# 1. make_robust_worker_node(name, worker_fn, retries): Node-Fabrik mit Retry-Logik und Fallback
# 2. Mit einer instabilen Worker-Funktion testen: Erfolgsfall und erzwungener Fehlerfall
# 3. Status (ok/fallback) und ggf. Fehlerliste im Rückgabe-State festhalten

**Praxis-Transfer: Meeting- & Briefing-Agent**

MVP-Canvas für den Meeting- & Briefing-Agent.

1. Welche reale Arbeitsfrage löst **dieser** Notebook-Baustein im Meeting- & Briefing-Agent?
2. Welche Eingabedaten oder Dokumente braucht er?
3. Welche Ausgabe sollte er liefern?
4. Welche Risiken oder Grenzen bleiben?
5. Wann wäre Human Review nötig?
6. Ist der passende Lösungsweg hier Prompt, strukturierte Ausgabe, RAG, Tool, Agent oder Workflow?

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [Checkliste Automatisierung](https://editor.p5js.org/ralf.bendig.rb/full/ckiLlKrql)
- [KI-Delegation-Poker](https://ki-delegation-poker.ralf42-2025.chatgpt.site/teilnehmen)
- [KI-Prozessoptimierung](https://editor.p5js.org/ralf.bendig.rb/full/xGKXCTR7T)
- [KI-Agenten-Architektur](https://editor.p5js.org/ralf.bendig.rb/full/Viso2emNI)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
- [Minimum Viable Agent Stack](https://ralf-42.github.io/Agenten/08-deployment-betrieb/minimum-viable-agent-stack.html)
- [Code Standards](https://ralf-42.github.io/Agenten/10-ressourcen/standards.html)
